# AI 기반 금융권 IT감사 사전 통제 점검 분석
> 법령 기반 Rule 엔진 위반 탐지 결과 분석

## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Mac: AppleGothic / Windows: Malgun Gothic)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

BASE = '../data'
print('환경 설정 완료')

## 1. 데이터 로드

In [ ]:
df_emp     = pd.read_csv(f'{BASE}/processed/virtual_db/emp_master.csv',   encoding='utf-8-sig')
df_account = pd.read_csv(f'{BASE}/processed/virtual_db/sys_account.csv',  encoding='utf-8-sig')
df_access  = pd.read_csv(f'{BASE}/processed/virtual_db/access_log.csv',   encoding='utf-8-sig')
df_itsm    = pd.read_csv(f'{BASE}/processed/virtual_db/itsm_req.csv',     encoding='utf-8-sig')
df_deploy  = pd.read_csv(f'{BASE}/processed/virtual_db/deploy_log.csv',   encoding='utf-8-sig')
df_backup  = pd.read_csv(f'{BASE}/processed/virtual_db/backup_log.csv',   encoding='utf-8-sig')
df_violations = pd.read_csv(f'{BASE}/processed/violations_summary.csv',   encoding='utf-8-sig')

for df, cols in [
    (df_emp,     ['hire_dt','resign_dt']),
    (df_account, ['last_review_dt','last_pw_change_dt','create_dt']),
    (df_access,  ['access_dt']),
    (df_deploy,  ['deploy_dt']),
    (df_itsm,    ['request_dt','approval_dt']),
    (df_backup,  ['backup_dt']),
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

print('데이터 로드 완료')
for name, df in [('emp_master',df_emp),('sys_account',df_account),('access_log',df_access),
                  ('itsm_req',df_itsm),('deploy_log',df_deploy),('backup_log',df_backup),('violations',df_violations)]:
    print(f'  {name:15s}: {len(df):,}행')

## 2. 전체 위반 현황 요약

In [ ]:
total    = len(df_violations)
violated = (df_violations['yn_violation'] == 'Y').sum()
clean    = total - violated

print('='*45)
print('  IT감사 Rule 엔진 점검 결과')
print('='*45)
print(f'  총 점검 규칙 : {total}개')
print(f'  위반 탐지    : {violated}개 ({violated/total*100:.1f}%)')
print(f'  이상 없음    : {clean}개 ({clean/total*100:.1f}%)')

domain_summary = df_violations.groupby('audit_domain').agg(
    전체규칙=('rule_id','count'),
    위반규칙=('yn_violation', lambda x: (x=='Y').sum()),
    총위반건수=('violation_count','sum')
).reset_index()
domain_summary['위반율(%)'] = (domain_summary['위반규칙']/domain_summary['전체규칙']*100).round(1)
print()
print('[도메인별 위반 현황]')
print(domain_summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ① 도메인별 위반 규칙 수
colors = ['#e74c3c','#3498db','#2ecc71']
axes[0].bar(domain_summary['audit_domain'], domain_summary['위반규칙'], color=colors, width=0.5)
axes[0].set_title('도메인별 위반 탐지 규칙 수', fontsize=12, fontweight='bold')
axes[0].set_ylabel('규칙 수')
for i,v in enumerate(domain_summary['위반규칙']):
    axes[0].text(i, v+0.2, f'{v}개', ha='center', fontweight='bold')

# ② 심각도별 위반 규칙 수
sev_colors = {'HIGH':'#e74c3c','MEDIUM':'#f39c12','LOW':'#3498db'}
sev_data = df_violations[df_violations['yn_violation']=='Y'].groupby('severity').size().reindex(['HIGH','MEDIUM','LOW'],fill_value=0)
bars = axes[1].bar(sev_data.index, sev_data.values, color=[sev_colors[s] for s in sev_data.index], width=0.5)
axes[1].set_title('심각도별 위반 규칙 수', fontsize=12, fontweight='bold')
axes[1].set_ylabel('규칙 수')
for i,v in enumerate(sev_data.values):
    axes[1].text(i, v+0.2, f'{v}개', ha='center', fontweight='bold')

# ③ 위반/정상 파이차트
lbl1 = f'위반 {violated}개'
lbl2 = f'이상없음 {clean}개'
axes[2].pie([violated, clean], labels=[lbl1, lbl2],
            colors=['#e74c3c','#2ecc71'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize':11})
axes[2].set_title('전체 규칙 위반 비율', fontsize=12, fontweight='bold')

plt.suptitle('IT감사 Rule 엔진 점검 결과 종합', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. 접근통제 분석

In [ ]:
# 3-1. 퇴사자 계정 미정리
resigned_ids = df_emp[df_emp['yn_employed']=='N']['emp_id'].tolist()
active_resigned = df_account[
    df_account['emp_id'].isin(resigned_ids) & (df_account['account_status']=='active')
].copy().merge(df_emp[['emp_id','emp_nm','dept_nm','resign_dt']], on='emp_id', how='left')

print(f'퇴사자 활성 계정: {len(active_resigned)}건')
active_resigned['방치기간(일)'] = (pd.Timestamp('2026-04-30') - active_resigned['resign_dt']).dt.days
print(active_resigned[['emp_nm','dept_nm','system_cd','resign_dt','방치기간(일)']].sort_values('방치기간(일)',ascending=False).to_string(index=False))

In [ ]:
# 3-2. 권한검토 초과 계정
overdue = df_account[df_account['yn_overdue_review']=='Y'].copy()
overdue = overdue.merge(df_emp[['emp_id','dept_nm','role_type']], on='emp_id', how='left')
print(f'권한검토 180일 초과: {len(overdue)}건')
print()
print('[부서별 현황]')
print(overdue.groupby('dept_nm').size().sort_values(ascending=False).to_string())

In [ ]:
# 3-3. 업무시간 외 접속 현황
after_hours = df_access[df_access['yn_after_hours']=='Y'].copy()
after_hours['hour'] = after_hours['access_dt'].dt.hour
after_hours['weekday'] = after_hours['access_dt'].dt.day_name()

print(f'업무시간 외(22~06시) 접속: {len(after_hours):,}건')
print()
print('[시스템별]')
print(after_hours.groupby('system_cd').size().sort_values(ascending=False).to_string())
print()
print('[액션 유형별]')
print(after_hours.groupby('action_type').size().sort_values(ascending=False).to_string())

In [ ]:
# 3-4. 업무시간 외 접속 - 요일×시간 히트맵
after_hours['hour'] = after_hours['access_dt'].dt.hour
after_hours['요일'] = after_hours['access_dt'].dt.dayofweek  # 0=월요일
weekday_map = {0:'월',1:'화',2:'수',3:'목',4:'금',5:'토',6:'일'}
after_hours['요일명'] = after_hours['요일'].map(weekday_map)

pivot = after_hours.groupby(['요일','hour']).size().unstack(fill_value=0)
pivot.index = [weekday_map[i] for i in pivot.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# 히트맵
import matplotlib.colors as mcolors
im = axes[0].imshow(pivot.values, cmap='Reds', aspect='auto')
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels([f'{h}시' for h in pivot.columns], fontsize=8)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index)
axes[0].set_title('요일×시간대별 업무시간 외 접속 히트맵', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0], label='접속 건수')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i,j]
        if v > 0:
            axes[0].text(j, i, str(v), ha='center', va='center', fontsize=8,
                        color='white' if v > pivot.values.max()*0.6 else 'black')

# 시간대 분포 막대
after_hours['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#e74c3c', alpha=0.8)
axes[1].set_title('시간대별 업무시간 외 접속 건수', fontsize=12, fontweight='bold')
axes[1].set_xlabel('시각')
axes[1].set_ylabel('건수')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 4. 변경관리 분석

In [ ]:
no_cr    = df_deploy[df_deploy['yn_no_cr']=='Y']
post_app = df_deploy[df_deploy['yn_post_approval']=='Y']
job_sep  = df_deploy[df_deploy['yn_job_sep_violation']=='Y']

print('[변경관리 위반 현황]')
print(f'  CR 없는 무단 배포        : {len(no_cr):>3}건')
print(f'  사후 승인 배포           : {len(post_app):>3}건')
print(f'  직무분리 위반 (신청=배포) : {len(job_sep):>3}건')

In [ ]:
# 4-2. 변경관리 위반 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ① 위반 유형별 건수
viol_types = ['CR 없는 무단배포', '사후 승인 배포', '직무분리 위반']
viol_counts = [len(no_cr), len(post_app), len(job_sep)]
colors_cm = ['#e74c3c','#f39c12','#9b59b6']
bars = axes[0].bar(viol_types, viol_counts, color=colors_cm, width=0.5)
axes[0].set_title('변경관리 위반 유형별 건수', fontsize=12, fontweight='bold')
axes[0].set_ylabel('건수')
for bar, v in zip(bars, viol_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v}건',
                ha='center', fontweight='bold')

# ② 사후 승인 지연 시간 분포
pa = post_app.merge(df_itsm[['doc_no','approval_dt']], on='doc_no', how='left')
pa['delay_h'] = (pa['approval_dt'] - pa['deploy_dt']).dt.total_seconds()/3600
axes[1].hist(pa['delay_h'].dropna(), bins=10, color='#f39c12', edgecolor='white', alpha=0.85)
axes[1].axvline(pa['delay_h'].mean(), color='red', linestyle='--', label=f"평균 {pa['delay_h'].mean():.1f}h")
axes[1].set_title('사후승인 배포: 배포→승인 소요 시간', fontsize=12, fontweight='bold')
axes[1].set_xlabel('시간(h)')
axes[1].set_ylabel('건수')
axes[1].legend()

# ③ 월별 배포 건수 (정상 vs 위반)
df_deploy['월'] = df_deploy['deploy_dt'].dt.to_period('M').astype(str)
monthly_total = df_deploy.groupby('월').size()
monthly_viol  = df_deploy[df_deploy['yn_no_cr']=='Y'].groupby('월').size()
monthly_viol  = monthly_viol.reindex(monthly_total.index, fill_value=0)
x = range(len(monthly_total))
axes[2].bar(x, monthly_total, label='전체 배포', color='#3498db', alpha=0.7)
axes[2].bar(x, monthly_viol,  label='무단 배포', color='#e74c3c', alpha=0.9)
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(monthly_total.index, rotation=45, ha='right', fontsize=8)
axes[2].set_title('월별 배포 건수 (정상 vs 무단)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('건수')
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. 운영통제 분석

In [ ]:
fail_backup = df_backup[df_backup['backup_status']=='F']
no_restore  = df_backup[(df_backup['backup_type'].str.upper()=='FULL') & (df_backup['yn_restore_test']=='N')]
print(f'백업 실패         : {len(fail_backup)}건')
print(f'복구테스트 미실시  : {len(no_restore)}건')
print()
print('[시스템별 백업 실패]')
print(fail_backup.groupby('system_cd').size().sort_values(ascending=False).to_string())

In [ ]:
# 5-2. 시스템별 백업 성공률 + 복구테스트율
backup_rate = df_backup.groupby('system_cd').apply(
    lambda x: pd.Series({
        '백업성공률': (x['backup_status']=='S').mean()*100,
        '복구테스트율': (x[x['backup_type'].str.upper()=='FULL']['yn_restore_test']=='Y').mean()*100
            if len(x[x['backup_type'].str.upper()=='FULL'])>0 else 0
    })
).round(1).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 백업 성공률
colors_b = ['#e74c3c' if v < 97 else '#2ecc71' for v in backup_rate['백업성공률']]
axes[0].bar(backup_rate['system_cd'], backup_rate['백업성공률'], color=colors_b)
axes[0].set_title('시스템별 백업 성공률', fontsize=12, fontweight='bold')
axes[0].set_ylabel('성공률 (%)')
axes[0].set_ylim(90, 102)
axes[0].axhline(97, color='red', linestyle='--', alpha=0.5, label='기준선(97%)')
for i, v in enumerate(backup_rate['백업성공률']):
    axes[0].text(i, v+0.2, f'{v}%', ha='center', fontsize=9)
axes[0].legend()

# 복구테스트율
colors_r = ['#e74c3c' if v < 80 else '#f39c12' if v < 100 else '#2ecc71' for v in backup_rate['복구테스트율']]
axes[1].bar(backup_rate['system_cd'], backup_rate['복구테스트율'], color=colors_r)
axes[1].set_title('시스템별 FULL백업 복구테스트 수행율', fontsize=12, fontweight='bold')
axes[1].set_ylabel('수행율 (%)')
axes[1].set_ylim(0, 110)
for i, v in enumerate(backup_rate['복구테스트율']):
    axes[1].text(i, v+1, f'{v}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 6. 심화 분석 및 인사이트

### 6-1. 부서별 위험도 히트맵

In [ ]:
# 부서 × 위반유형 교차 히트맵
resigned_ids = set(df_emp[df_emp['yn_employed']=='N']['emp_id'])
active_resigned = df_account[
    df_account['emp_id'].isin(resigned_ids) & (df_account['account_status']=='active')
].copy().merge(df_emp[['emp_id','dept_nm']], on='emp_id', how='left')

overdue = df_account[df_account['yn_overdue_review']=='Y'].merge(
    df_emp[['emp_id','dept_nm']], on='emp_id', how='left')

after_hours = df_access[df_access['yn_after_hours']=='Y'].merge(
    df_emp[['emp_id','dept_nm']], on='emp_id', how='left')

risk_resign  = active_resigned.groupby('dept_nm').size().rename('퇴사자계정')
risk_overdue = overdue.groupby('dept_nm').size().rename('권한검토초과')
risk_after   = after_hours.groupby('dept_nm').size().rename('시간외접속')

dept_risk = pd.concat([risk_resign, risk_overdue, risk_after], axis=1).fillna(0).astype(int)
dept_risk['위험점수'] = dept_risk['퇴사자계정']*3 + dept_risk['권한검토초과']*2 + dept_risk['시간외접속']
dept_risk = dept_risk.sort_values('위험점수', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 히트맵
heat_data = dept_risk[['퇴사자계정','권한검토초과','시간외접속']].head(12)
im = axes[0].imshow(heat_data.values, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(3))
axes[0].set_xticklabels(['퇴사자계정', '권한검토초과', '시간외접속'])
axes[0].set_yticks(range(len(heat_data)))
axes[0].set_yticklabels(heat_data.index, fontsize=9)
axes[0].set_title('부서별 접근통제 위반 히트맵', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0], label='위반 건수')
for i in range(len(heat_data)):
    for j in range(3):
        v = heat_data.values[i,j]
        if v > 0:
            axes[0].text(j, i, str(int(v)), ha='center', va='center',
                        fontsize=9, color='white' if v > heat_data.values.max()*0.6 else 'black')

# 위험점수 수평 막대
top10 = dept_risk.head(10)
score_colors = ['#e74c3c' if s >= 10 else '#f39c12' if s >= 5 else '#3498db' for s in top10['위험점수']]
axes[1].barh(top10.index[::-1], top10['위험점수'][::-1], color=score_colors[::-1])
axes[1].set_title('부서별 종합 위험점수 TOP 10', fontsize=12, fontweight='bold')
axes[1].set_xlabel('위험점수 (퇴사자계정x3 + 권한검토초과x2 + 시간외접속x1)')
for i, v in enumerate(top10['위험점수'][::-1]):
    axes[1].text(v+0.1, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()
print(dept_risk.to_string())

### 6-2. 시스템별 위험도 분석

In [ ]:
# 시스템별 접근통제 위반 집중도
sys_after = df_access[df_access['yn_after_hours']=='Y'].groupby('system_cd').size().rename('시간외접속')
sys_fail  = df_access[df_access['result_cd']=='F'].groupby('system_cd').size().rename('로그인실패')
sys_resign = df_access[df_access['yn_post_resign_access']=='Y'].groupby('system_cd').size().rename('퇴사자접속')

sys_risk = pd.concat([sys_after, sys_fail, sys_resign], axis=1).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 시스템별 위반 유형 누적 막대
x = range(len(sys_risk))
axes[0].bar(x, sys_risk['시간외접속'],  label='시간외 접속',  color='#e74c3c', alpha=0.85)
axes[0].bar(x, sys_risk['로그인실패'],  label='로그인 실패',  color='#f39c12', alpha=0.85, bottom=sys_risk['시간외접속'])
axes[0].bar(x, sys_risk['퇴사자접속'],  label='퇴사자 접속',  color='#9b59b6', alpha=0.85,
            bottom=sys_risk['시간외접속']+sys_risk['로그인실패'])
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(sys_risk.index)
axes[0].set_title('시스템별 접근통제 위반 현황', fontsize=12, fontweight='bold')
axes[0].set_ylabel('건수')
axes[0].legend()

# 시스템별 백업 위험도
sys_backup = df_backup.groupby('system_cd').apply(
    lambda x: pd.Series({
        '백업실패건수': (x['backup_status']=='F').sum(),
        '복구미실시': (x[x['backup_type'].str.upper()=='FULL']['yn_restore_test']=='N').sum()
            if len(x[x['backup_type'].str.upper()=='FULL'])>0 else 0
    })
).reset_index()
x2 = range(len(sys_backup))
axes[1].bar([i-0.2 for i in x2], sys_backup['백업실패건수'], width=0.4, label='백업 실패', color='#e74c3c', alpha=0.85)
axes[1].bar([i+0.2 for i in x2], sys_backup['복구미실시'],  width=0.4, label='복구테스트 미실시', color='#f39c12', alpha=0.85)
axes[1].set_xticks(list(x2))
axes[1].set_xticklabels(sys_backup['system_cd'])
axes[1].set_title('시스템별 운영통제 위험 현황', fontsize=12, fontweight='bold')
axes[1].set_ylabel('건수')
axes[1].legend()

plt.tight_layout()
plt.show()

### 6-3. 역할(Role)별 위반 분석

In [ ]:
# role_type별 위반 집중도
role_map = df_emp[['emp_id','role_type','dept_nm']].copy()

# 업무시간외 접속 - role별
ah_role = df_access[df_access['yn_after_hours']=='Y'].merge(role_map, on='emp_id', how='left')
ah_by_role = ah_role.groupby('role_type').size()

# 권한검토 초과 - role별
od_role = df_account[df_account['yn_overdue_review']=='Y'].merge(role_map, on='emp_id', how='left')
od_by_role = od_role.groupby('role_type').size()

# 퇴사자 활성계정 - role별
ar_role = active_resigned.merge(df_emp[['emp_id','role_type']], on='emp_id', how='left')
ar_by_role = ar_role.groupby('role_type').size() if 'role_type' in ar_role.columns else pd.Series(dtype=int)

roles = ['ADMIN','DEV','NORMAL','SECUR']
role_df = pd.DataFrame({
    '시간외접속':   [ah_by_role.get(r,0) for r in roles],
    '권한검토초과': [od_by_role.get(r,0) for r in roles],
    '퇴사자계정':   [ar_by_role.get(r,0) for r in roles],
}, index=roles)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# role별 위반 유형 grouped bar
x = np.arange(len(roles))
w = 0.25
axes[0].bar(x-w,   role_df['시간외접속'],   width=w, label='시간외 접속',  color='#e74c3c', alpha=0.85)
axes[0].bar(x,     role_df['권한검토초과'], width=w, label='권한검토 초과', color='#f39c12', alpha=0.85)
axes[0].bar(x+w,   role_df['퇴사자계정'],   width=w, label='퇴사자 계정',  color='#9b59b6', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(roles)
axes[0].set_title('역할(Role)별 접근통제 위반 현황', fontsize=12, fontweight='bold')
axes[0].set_ylabel('건수')
axes[0].legend()

# role별 총 직원 수 vs 위반자 수 비율
role_total = df_emp.groupby('role_type').size().reindex(roles, fill_value=0)
violators = set(
    list(ah_role['emp_id'].dropna().unique()) +
    list(od_role['emp_id'].dropna().unique())
)
role_viol_cnt = df_emp[df_emp['emp_id'].isin(violators)].groupby('role_type').size().reindex(roles, fill_value=0)
role_viol_rate = (role_viol_cnt / role_total * 100).round(1)

colors_r = ['#e74c3c' if v>20 else '#f39c12' if v>10 else '#3498db' for v in role_viol_rate]
axes[1].bar(roles, role_viol_rate, color=colors_r, width=0.5)
axes[1].set_title('역할별 위반 관여 비율', fontsize=12, fontweight='bold')
axes[1].set_ylabel('비율 (%)')
for i, v in enumerate(role_viol_rate):
    axes[1].text(i, v+0.5, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print()
print('[역할별 위반 현황 요약]')
print(pd.DataFrame({'전체인원':role_total,'위반관여':role_viol_cnt,'위반율(%)':role_viol_rate}).to_string())

### 6-4. 레이더 차트 - 도메인별 위험도

In [ ]:
# 레이더 차트: 도메인 × 위험 항목별 점수
import numpy as np

categories = ['퇴사자계정\n미삭제', '권한검토\n초과', '시간외\n접속', '무단배포', '사후승인\n배포', '직무분리\n위반', '백업실패', '복구테스트\n미실시']
# 각 항목 실제 위반 건수 (최대값으로 정규화)
raw_values = [
    len(active_resigned),
    len(overdue),
    len(after_hours),
    len(df_deploy[df_deploy['yn_no_cr']=='Y']),
    len(df_deploy[df_deploy['yn_post_approval']=='Y']),
    len(df_deploy[df_deploy['yn_job_sep_violation']=='Y']),
    len(df_backup[df_backup['backup_status']=='F']),
    len(df_backup[(df_backup['backup_type'].str.upper()=='FULL') & (df_backup['yn_restore_test']=='N')])
]
max_val = max(raw_values)
values = [v/max_val*100 for v in raw_values]
values += values[:1]  # 닫기

N = len(categories)
angles = [n/float(N)*2*np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, values, 'o-', linewidth=2, color='#e74c3c')
ax.fill(angles, values, alpha=0.25, color='#e74c3c')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20','40','60','80','100'], size=8)
ax.set_title('IT감사 위반 항목별 위험도 레이더 차트\n(최대 위반 건수 대비 상대값)', fontsize=13, fontweight='bold', pad=20)

# 실제 건수 주석
for angle, value, label, raw in zip(angles[:-1], values[:-1], categories, raw_values):
    ax.annotate(f'{raw}건', xy=(angle, value), xytext=(angle, value+8),
                ha='center', fontsize=9, color='#c0392b', fontweight='bold')

plt.tight_layout()
plt.show()

### 6-5. 법령별 위반 현황

In [ ]:
law_viol = df_violations[df_violations['yn_violation']=='Y'].copy()
law_summary = law_viol.groupby('source_law').agg(
    위반규칙수=('rule_id','count'),
    총위반건수=('violation_count','sum'),
    HIGH건수=('severity', lambda x: (x=='HIGH').sum())
).reset_index().sort_values('위반규칙수', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 법령별 위반 규칙 수 (수평 막대)
colors_law = ['#e74c3c' if h>0 else '#3498db' for h in law_summary['HIGH건수']]
bars = axes[0].barh(law_summary['source_law'], law_summary['위반규칙수'], color=colors_law)
axes[0].set_title('법령별 위반 탐지 규칙 수\n(빨간색: HIGH 위반 포함)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('위반 규칙 수')
for i, (v, h) in enumerate(zip(law_summary['위반규칙수'], law_summary['HIGH건수'])):
    label = f'{v}개 (HIGH {h}건)' if h>0 else f'{v}개'
    axes[0].text(v+0.05, i, label, va='center', fontsize=9)
high_patch = mpatches.Patch(color='#e74c3c', label='HIGH 위반 포함')
norm_patch = mpatches.Patch(color='#3498db', label='MEDIUM/LOW')
axes[0].legend(handles=[high_patch, norm_patch])

# 법령별 준수율 (위반 없는 규칙 비율)
law_total = df_violations.groupby('source_law').size()
law_viol_cnt = df_violations[df_violations['yn_violation']=='Y'].groupby('source_law').size()
law_compliance = ((1 - law_viol_cnt/law_total)*100).round(1).reset_index()
law_compliance.columns = ['source_law','준수율(%)']
law_compliance = law_compliance.sort_values('준수율(%)')

colors_c = ['#e74c3c' if v<50 else '#f39c12' if v<75 else '#2ecc71' for v in law_compliance['준수율(%)']]
axes[1].barh(law_compliance['source_law'], law_compliance['준수율(%)'], color=colors_c)
axes[1].set_title('법령별 규칙 준수율', fontsize=12, fontweight='bold')
axes[1].set_xlabel('준수율 (%)')
axes[1].set_xlim(0, 115)
for i, v in enumerate(law_compliance['준수율(%)']):
    axes[1].text(v+0.5, i, f'{v}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()
print()
print(law_summary.to_string(index=False))

### 6-6. 복합 위반자 분석

In [ ]:
resigned_active_emp = set(active_resigned['emp_id'])
after_hours_emp     = set(after_hours['emp_id'].dropna())
overdue_emp         = set(overdue['emp_id'])

def get_flags(emp_id):
    flags = []
    if emp_id in resigned_active_emp: flags.append('퇴사자계정')
    if emp_id in after_hours_emp:     flags.append('시간외접속')
    if emp_id in overdue_emp:         flags.append('권한검토초과')
    return flags

all_viol_emp = resigned_active_emp | after_hours_emp | overdue_emp
risk_persons = pd.DataFrame({'emp_id': list(all_viol_emp)})
risk_persons['위반목록'] = risk_persons['emp_id'].apply(lambda x: ', '.join(get_flags(x)))
risk_persons['위반수']   = risk_persons['emp_id'].apply(lambda x: len(get_flags(x)))
risk_persons = risk_persons.merge(df_emp[['emp_id','emp_nm','dept_nm','role_type','yn_employed']], on='emp_id', how='left')
risk_persons = risk_persons.sort_values('위반수', ascending=False)

multi = risk_persons[risk_persons['위반수'] >= 2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 위반 수별 인원 분포
viol_count_dist = risk_persons['위반수'].value_counts().sort_index()
axes[0].bar(viol_count_dist.index.astype(str)+'개 위반', viol_count_dist.values,
            color=['#3498db','#f39c12','#e74c3c'][:len(viol_count_dist)], width=0.5)
axes[0].set_title('위반 항목 수별 인원 분포', fontsize=12, fontweight='bold')
axes[0].set_ylabel('인원 수')
for i, v in enumerate(viol_count_dist.values):
    axes[0].text(i, v+0.2, f'{v}명', ha='center', fontweight='bold')

# 복합 위반자 부서 분포
if len(multi) > 0:
    multi_dept = multi.groupby('dept_nm').size().sort_values(ascending=True)
    axes[1].barh(multi_dept.index, multi_dept.values, color='#e74c3c', alpha=0.85)
    axes[1].set_title(f'복합 위반자 ({len(multi)}명) 부서 분포', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('인원 수')
    for i, v in enumerate(multi_dept.values):
        axes[1].text(v+0.05, i, str(v), va='center')

plt.tight_layout()
plt.show()
print(f'\n복합 위반자 (2개 이상): {len(multi)}명')
print(multi[['emp_nm','dept_nm','role_type','yn_employed','위반목록','위반수']].to_string(index=False))

### 6-7. 핵심 발견사항 종합 요약

In [ ]:
print('=' * 62)
print('  IT감사 핵심 발견사항 요약')
print('=' * 62)

total_r    = len(df_violations)
violated_r = (df_violations['yn_violation']=='Y').sum()

print(f'''
점검 기간  : 2025년 11월 ~ 2026년 4월 (6개월)
점검 규칙  : {total_r}개 | 위반 탐지 : {violated_r}개 ({violated_r/total_r*100:.1f}%)

【접근통제 분야 - 위험도: 高】
 1. 퇴사자 계정 미삭제 : {len(active_resigned)}건
    → 퇴사 후 최장 {active_resigned["방치기간(일)"].max() if "방치기간(일)" in active_resigned.columns else "N/A"}일 이상 활성 계정 유지
    → 전자금융감독규정 제29조(접근권한 관리) 위반

 2. 권한검토 180일 초과 : {len(overdue)}건
    → 영업1팀(5건), IT개발팀(6건) 집중
    → 주기적 권한검토 의무 미이행

 3. 업무시간 외 접속 : {len(after_hours)}건
    → UPDATE·DOWNLOAD 등 고위험 액션 포함
    → 이상거래 여부 추가 검토 필요

【변경관리 분야 - 위험도: 中高】
 4. 무단 배포(CR 없음) : {len(df_deploy[df_deploy["yn_no_cr"]=="Y"])}건
 5. 사후 승인 배포     : {len(df_deploy[df_deploy["yn_post_approval"]=="Y"])}건 (배포 후 평균 {pa["delay_h"].mean():.1f}시간 후 승인)
 6. 직무분리 위반      : {len(df_deploy[df_deploy["yn_job_sep_violation"]=="Y"])}건

【운영통제 분야 - 위험도: 中】
 7. 백업 실패          : {len(fail_backup)}건
 8. 복구테스트 미실시  : {len(no_restore)}건
''')
print('=' * 62)